# 01 - Generate the single-transistor datasets (Part A, step 1)

Self-contained notebook: everything it needs is defined below. The only external inputs are the model file, ngspice, and the usual
Python libraries (numpy, pandas, matplotlib).

For each of the four cases (NMOS ON, NMOS OFF, PMOS ON, PMOS OFF) it draws 10,000 random samples (section 4 of the spec),
simulates each one in a one-FET ngspice netlist and stores one CSV per case.
Columns: `vdd, temp, w, vd, vs, vg, vb, vth0, toxe, ndep, u0, xj, id, ig, is, ib`. Currents are terminal currents flowing **into** the device.

## 1. Configuration
Assignment: model file `16nm_HP`, nominal VDD 0.9 V. Everything else follows the sweep ranges of the spec.

In [1]:
import re
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# ---- paths: the notebook lives in notebooks/; outputs go to the folders next to it
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_FILE = ROOT / "Model Files-20261105" / "models" / "16nm_HP.pm"     # assigned model file
BUILD_DIR = ROOT / "build"          # scratch space for generated netlists (git-ignored)
NETLIST_DIR = ROOT / "netlists"
DATA_DIR = ROOT / "data"

# ---- assignment and sweep ranges (project spec, section 4)
VDD_NOM = 0.9
LMIN = 16e-9                        # channel length = feature size of the node
WMIN = LMIN
W_RANGE = (1 * LMIN, 8 * LMIN)      # transistor width, uniform
LDIFF = 2 * LMIN                    # diffusion length for the junction areas AD = AS = W * LDIFF
VDD_VAR = 0.10                      # VDD uniform in nominal +/- 10 %
TEMP_RANGE = (-40.0, 125.0)         # degC, uniform
PROC_VAR = 0.10                     # process parameters uniform in nominal +/- 10 %
PROC_PARAMS = ("vth0", "toxe", "ndep", "u0", "xj")

N_SAMPLES = 10_000
SEED = 20261105
# gate and body voltage per case, as a fraction of the sampled VDD
CASES = {
    "nmos_on":  dict(device="nmos", vg=1.0, vb=0.0),
    "nmos_off": dict(device="nmos", vg=0.0, vb=0.0),
    "pmos_on":  dict(device="pmos", vg=0.0, vb=1.0),
    "pmos_off": dict(device="pmos", vg=1.0, vb=1.0),
}
CASE_SEEDS = {name: SEED + i for i, name in enumerate(CASES)}

SPICE_OPTIONS = ".options gmin=1e-18 abstol=1e-18 reltol=1e-6 vntol=1e-9"   # required in every netlist
NUMDGT = 15                         # printed digits; the default 6 breaks the Kirchhoff check
N_WORKERS = 8                       # parallel ngspice processes
CHUNK = 250                         # samples per ngspice process
CURRENTS = ["id", "ig", "is", "ib"]
print(f"model {MODEL_FILE.name} | exists: {MODEL_FILE.exists()} | nominal VDD {VDD_NOM} V | L = Wmin = {LMIN*1e9:.0f} nm")

model 16nm_HP.pm | exists: True | nominal VDD 0.9 V | L = Wmin = 16 nm


## 2. Nominal process parameters from the model file
Read straight from the `.model` cards; every sample is `nominal x U(0.9, 1.1)`, separately for the NMOS and PMOS cards.

In [2]:
def read_cards(path):
    """Return {card_name: text}; continuation lines ('+') are joined to their .model line."""
    cards, cur = {}, None
    for raw in Path(path).read_text().splitlines():
        s = raw.strip()
        if not s or s.startswith("*"):
            continue
        if s.lower().startswith(".model"):
            cur = s.split()[1].lower()
            cards[cur] = [s]
        elif s.startswith("+") and cur:
            cards[cur].append(s[1:])
        else:
            cur = None
    return {k: " ".join(v) for k, v in cards.items()}


_NUM = r"[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?"


def nominal_values(path, params):
    """Return {card_name: {param: value}} for the requested parameters."""
    out = {}
    for name, text in read_cards(path).items():
        vals = {}
        for p in params:
            # (?<![\w.]) keeps e.g. 'vth0' from matching inside 'lvth0' or 'wvth0'
            m = re.search(rf"(?<![\w.]){p}\s*=\s*({_NUM})", text, re.I)
            if m is None:
                raise ValueError(f"parameter {p} not found in card {name}")
            vals[p] = float(m.group(1))
        out[name] = vals
    return out


def nominals():
    return nominal_values(MODEL_FILE, PROC_PARAMS)


pd.DataFrame(nominals()).T

,vth0,toxe,ndep,u0,xj
nmos,0.47965,9.500000e-10,7.000000e+18,0.030,5.000000e-09
pmos,-0.43121,1.000000e-09,5.500000e+18,0.006,5.000000e-09


## 3. ngspice wrapper
Runs ngspice natively if it is on the PATH, otherwise through WSL (Windows). Only stdout is parsed: ngspice writes progress messages to stderr,
and when both streams are merged they can cut result lines in half.

In [3]:
def to_wsl(path):
    """Convert a Windows path to its /mnt/<drive>/... form."""
    p = Path(path).resolve()
    return "/mnt/" + p.drive.rstrip(":").lower() + "/" + "/".join(p.parts[1:])


def prepare_build():
    """Create the scratch directory and copy the model file there (no spaces in the path)."""
    BUILD_DIR.mkdir(exist_ok=True)
    dst = BUILD_DIR / MODEL_FILE.name
    if not dst.exists() or dst.read_bytes() != MODEL_FILE.read_bytes():
        shutil.copyfile(MODEL_FILE, dst)
    return BUILD_DIR


def run_ngspice(cir_name, cwd, merge_stderr=False):
    """Run `ngspice -b cir_name` inside cwd and return its output as text."""
    cwd = Path(cwd)
    if shutil.which("ngspice"):
        r = subprocess.run(["ngspice", "-b", str(cir_name)], cwd=cwd, capture_output=True, text=True, errors="replace")
        return (r.stdout + (r.stderr if merge_stderr else "")).replace("\x00", "")
    cmd = f"cd '{to_wsl(cwd)}' && ngspice -b '{cir_name}'" + (" 2>&1" if merge_stderr else " 2>/dev/null")
    r = subprocess.run(["wsl", "-e", "bash", "-c", cmd], capture_output=True, text=True, errors="replace")
    return r.stdout.replace("\x00", "")


def ngspice_version():
    if shutil.which("ngspice"):
        r = subprocess.run(["ngspice", "--version"], capture_output=True, text=True, errors="replace")
        text = r.stdout + r.stderr
    else:
        r = subprocess.run(["wsl", "-e", "bash", "-c", "ngspice --version 2>&1"], capture_output=True, text=True, errors="replace")
        text = r.stdout.replace("\x00", "")
    m = re.search(r"ngspice-\d+(?:\.\d+)?", text)
    return m.group(0) if m else "unknown"


print("simulator:", ngspice_version())

simulator: ngspice-42


## 4. Netlists
* `example_netlist`: one readable netlist per case (nominal process, one voltage source per terminal, `AD = AS = W x Ldiff`) written to `netlists/`.
* `batch_netlist`: an explicit script that simulates many samples in one ngspice run. Per sample it sets the five process parameters
  (**one `altermod` per parameter**: only the first parameter on an `altermod` line is applied), the temperature, the four bias voltages,
  `W`, `AD` and `AS`, runs `op` and prints the terminal currents plus the node voltages. The node voltages let the parser detect a sample whose operating point failed.

In [4]:
def fmt(x):
    """Number formatting shared by the netlists and the CSV files (10 significant digits)."""
    return f"{float(x):.10g}"


def example_netlist(case, w=None, temp=27.0):
    spec = CASES[case]
    dev = spec["device"]
    w = WMIN if w is None else w
    vdd = VDD_NOM
    vd, vs = (vdd, 0.0) if dev == "nmos" else (0.0, vdd)
    vg, vb = spec["vg"] * vdd, spec["vb"] * vdd
    nom = nominals()[dev]
    inc = f"../{MODEL_FILE.parent.parent.name}/{MODEL_FILE.parent.name}/{MODEL_FILE.name}"
    name = "Mn" if dev == "nmos" else "Mp"
    lines = [
        f"* Single-transistor netlist: {case.replace('_', ' ').upper()} ({MODEL_FILE.stem}, nominal process)",
        f'.include "{inc}"',
        SPICE_OPTIONS,
        "",
        "* geometry and bias of this example (the datasets sample these and the process parameters at random)",
        f".param Lmin={fmt(LMIN)}",
        f".param Ldiff={fmt(LDIFF)}",
        f".param Wd={fmt(w)}",
        f".param vdd={fmt(vdd)}",
        "",
        "* device with junction areas AD = AS = W*Ldiff, one voltage source per terminal",
        f"{name} d g s b {dev} W={{Wd}} L={{Lmin}} AD={{Wd*Ldiff}} AS={{Wd*Ldiff}}",
        f"Vd d 0 {fmt(vd)}",
        f"Vg g 0 {fmt(vg)}",
        f"Vs s 0 {fmt(vs)}",
        f"Vb b 0 {fmt(vb)}",
        "",
        ".control",
        f"set numdgt = {NUMDGT}",
        f"set temp = {fmt(temp)}",
        "* one altermod per parameter (only the first parameter on a line is applied)",
    ]
    lines += [f"altermod {dev} {p}={fmt(nom[p])}" for p in PROC_PARAMS]
    lines += ["op", "print i(vd) i(vg) i(vs) i(vb)", ".endc", ".end", ""]
    return "\n".join(lines)


def batch_netlist(case, df):
    """Explicit script that simulates every row of df; each sample is announced with an `@@ k` marker."""
    dev = CASES[case]["device"]
    r0 = df.iloc[0]
    lines = [
        f"* single-FET batch: {case}",
        f".include {MODEL_FILE.name}",
        SPICE_OPTIONS,
        f"M1 d g s b {dev} W={fmt(r0.w)} L={fmt(LMIN)} AD={fmt(r0.w * LDIFF)} AS={fmt(r0.w * LDIFF)}",
        f"Vd d 0 {fmt(r0.vd)}",
        f"Vg g 0 {fmt(r0.vg)}",
        f"Vs s 0 {fmt(r0.vs)}",
        f"Vb b 0 {fmt(r0.vb)}",
        ".control",
        f"set numdgt = {NUMDGT}",
    ]
    for k, row in enumerate(df.itertuples(index=False)):
        lines.append(f"echo @@ {k}")
        lines += [f"altermod {dev} {p}={fmt(getattr(row, p))}" for p in PROC_PARAMS]
        lines += [
            f"set temp = {fmt(row.temp)}",
            f"alter vd = {fmt(row.vd)}",
            f"alter vg = {fmt(row.vg)}",
            f"alter vs = {fmt(row.vs)}",
            f"alter vb = {fmt(row.vb)}",
            f"alter @m1[w] = {fmt(row.w)}",
            f"alter @m1[ad] = {fmt(row.w * LDIFF)}",
            f"alter @m1[as] = {fmt(row.w * LDIFF)}",
            "op",
            "print i(vd) i(vg) i(vs) i(vb) v(d) v(g) v(s) v(b)",
        ]
    lines += [".endc", ".end", ""]
    return "\n".join(lines)


print(example_netlist("nmos_on"))

* Single-transistor netlist: NMOS ON (16nm_HP, nominal process)
.include "../Model Files-20261105/models/16nm_HP.pm"
.options gmin=1e-18 abstol=1e-18 reltol=1e-6 vntol=1e-9

* geometry and bias of this example (the datasets sample these and the process parameters at random)
.param Lmin=1.6e-08
.param Ldiff=3.2e-08
.param Wd=1.6e-08
.param vdd=0.9

* device with junction areas AD = AS = W*Ldiff, one voltage source per terminal
Mn d g s b nmos W={Wd} L={Lmin} AD={Wd*Ldiff} AS={Wd*Ldiff}
Vd d 0 0.9
Vg g 0 0.9
Vs s 0 0
Vb b 0 0

.control
set numdgt = 15
set temp = 27
* one altermod per parameter (only the first parameter on a line is applied)
altermod nmos vth0=0.47965
altermod nmos toxe=9.5e-10
altermod nmos ndep=7e+18
altermod nmos u0=0.03
altermod nmos xj=5e-09
op
print i(vd) i(vg) i(vs) i(vb)
.endc
.end



## 5. Sampler
Independent random draws per case. VDD is drawn first because Vd and Vs depend on it; Vd and Vs are uniform in [0, VDD];
the gate and body voltages are fixed by the case. Values are rounded to the 10 digits used in the netlists, so the CSV equals what was simulated.

In [5]:
COLUMNS = ["vdd", "temp", "w", "vd", "vs", "vg", "vb", *PROC_PARAMS]


def _round(a):
    return np.array([float(f"{x:.10g}") for x in a])


def draw(case, n, rng):
    spec = CASES[case]
    nom = nominals()[spec["device"]]
    vdd = _round(rng.uniform(VDD_NOM * (1 - VDD_VAR), VDD_NOM * (1 + VDD_VAR), n))
    vd = _round(rng.uniform(0.0, 1.0, n) * vdd)
    vs = _round(rng.uniform(0.0, 1.0, n) * vdd)
    w = _round(rng.uniform(*W_RANGE, n))
    temp = _round(rng.uniform(*TEMP_RANGE, n))
    cols = dict(vdd=vdd, temp=temp, w=w, vd=vd, vs=vs, vg=_round(spec["vg"] * vdd), vb=_round(spec["vb"] * vdd))
    for p in PROC_PARAMS:
        cols[p] = _round(nom[p] * rng.uniform(1 - PROC_VAR, 1 + PROC_VAR, n))
    return pd.DataFrame(cols)[COLUMNS]


draw("nmos_on", 5, np.random.default_rng(CASE_SEEDS["nmos_on"]))

,vdd,temp,w,vd,vs,vg,vb,vth0,toxe,ndep,u0,xj
0,0.947210,102.374556,5.369076e-08,0.172749,0.389591,0.947210,0.0,0.432768,8.801810e-10,7.515770e+18,0.028247,5.268979e-09
1,0.982782,101.045318,5.885693e-08,0.052095,0.836689,0.982782,0.0,0.518878,9.928629e-10,6.878196e+18,0.028219,5.474755e-09
2,0.979216,-36.282165,4.455253e-08,0.225320,0.511495,0.979216,0.0,0.442236,9.400940e-10,6.819136e+18,0.031843,5.112827e-09
3,0.907496,40.368048,8.644431e-08,0.668532,0.539466,0.907496,0.0,0.469688,9.344558e-10,7.206781e+18,0.032142,4.578763e-09
4,0.917868,75.813737,1.083371e-07,0.783207,0.333388,0.917868,0.0,0.459402,8.936648e-10,7.646390e+18,0.027544,5.096820e-09
